> **cellule 0**

# AloePri pas à pas — Qwen3-8B obfusqué, du papier à la pratique

**Document pédagogique et interactif** : chaque bloc du notebook implémente une
brique de la **Figure 2** du papier *Towards Privacy-Preserving LLM Inference
via Covariant Obfuscation* ([arXiv 2603.01499](https://arxiv.org/pdf/2603.01499)).
Les références citent les **sections, équations et algorithmes** du papier.

**La figure 2 — vue d'ensemble d'AloePri :**

![Figure 2 — Overview of AloePri](figure_2_aloepri.png)

*(Figure 2 du papier : en haut, l'**obfuscation offline du modèle** — les
transformations inversibles $\mathrm{KeyMat}\cdot\mathrm{InvKeyMat}=I$,
$\mathrm{Mask}\cdot\mathrm{InvMask}=I$,
$\mathrm{Perm}\cdot\mathrm{InvPerm}=I$ et
$\mathrm{Scaling}\cdot\mathrm{InvScaling}=I$ appliquées aux poids
$W_{\mathrm{embed}}$, $W_{\mathrm{head}}$, $W_{\mathrm{attn}}$ (q/k/v/o :
`Block & Head Perm`, `Rot`, `RoPE & Softmax`), $W_{\mathrm{ffn}}$ (`SiLU`,
`Scaling`, `Perm`) ; en bas, l'**inférence online** — le `Secret Vocab Mapping`
(permutation $\tau$) transforme les textes en ids permutés, traités par les
$L$ blocs transformer obfusqués, puis dépermutés côté client.)*

> **cellule 1**

> **Plan du notebook (correspondance avec le papier)**
>
> | Bloc | Référence papier | Brique de la figure |
> | :--- | :--- | :--- |
> | 1. Notations & menace | §3.2, §3.3 | — |
> | 2. KeyMat / InvKeyMat | **Algorithme 1**, §5.2.1 | $\mathrm{KeyMat}\cdot\mathrm{InvKeyMat}=I$ |
> | 3. Perm & mapping secret | §5.2.2, §5.3 | $\mathrm{Perm}\cdot\mathrm{InvPerm}=I$, `Secret Vocab Mapping` |
> | 4. Noise / Scaling | §5.2.2, §5.2.4 | $\mathrm{Scaling}\cdot\mathrm{InvScaling}=I$ |
> | 5. Block & Head Perm, Rot | **Algorithme 2**, §5.2.3 | `Block & Head Perm`, `Rot` |
> | 6. RoPE & Softmax, SiLU | §5.2.3, §5.2.4 | les points de non-linéarité |
> | 7. La chaîne complète (offline) | §5.2.2–5.2.4, Fig. 2 | l'obfuscation du modèle entier |
> | 9. Sécurité : FT, AloePri complet (h>0), VMA, ISA | §5.4, §6, §7 | le chaînage $\hat{P}/\hat{Q}$ (h>0) + les canaux ISA |

> **cellule 2**

### Comment exécuter ce notebook

- Les cellules de **démonstration** (blocs 2-6, miniature locale) tournent sur
  CPU, rapidement, sans coût.
- Les cellules **lourdes** (fine-tuning 8B, transform chaîné h>0, attaques)
  sont conditionnées par `RUN_HEAVY` : passer à `True` dans la cellule de
  configuration pour les exécuter (~1-2 h Modal, coût GPU).
- Les **clés** (`obfuscation_keys.json`) et le **tokenizer** restent côté client
  — le serveur ne voit jamais que des ids permutés (posture stricte).

In [ ]:
# cellule 3
import os, sys
# bootstrap : racine du worktree sur sys.path. Le kernel nbconvert démarre
# dans le dossier du notebook (notebooks/) ; on remonte jusqu'à trouver le
# package aloepri/ pour que les imports des cellules 1.5 et 2.1 fonctionnent
# quel que soit le point de lancement (nbconvert, Jupyter racine ou notebooks/).
for _ in range(6):
    if os.path.isdir(os.path.join(os.getcwd(), "aloepri")):
        break
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import torch

# Constantes de la procédure (valeurs du plan — à utiliser telles quelles)
MODEL = "Qwen/Qwen3-8B"
SEED = 0
ALPHA_E = 0.3
BETA = 8

# Drapeau : False → exécution locale rapide (sections 0-1 + branches "sauté") ;
# True → exécute réellement les cellules Modal (fine-tuning 9.1b,
# transform chaîné 9.2, attaques 9.3/9.4, précision 9.6). Passer à True
# uniquement avec une CLI Modal authentifiée.
RUN_HEAVY = True

In [ ]:
# cellule 4
import os

print(f"racine      {os.getcwd()}")
print(f"torch      {torch.__version__}")
print(f"numpy      {np.__version__}")
assert isinstance(RUN_HEAVY, bool), "RUN_HEAVY doit être un booléen"
print(f"RUN_HEAVY  {RUN_HEAVY} (booléen ✓)")

modal_cli = os.path.expanduser("~/modal-venv/bin/modal")
print(f"Modal CLI  {modal_cli} -> " + ("accessible ✓" if os.path.exists(modal_cli)
                                       else "introuvable (cellules RUN_HEAVY indisponibles)"))

> **cellule 5**

## 1. Notations et modèle de menace — §3.2, §3.3

**Notations** (tableau §3.3 du papier) :

| Symbole | Sens |
| :--- | :--- |
| $x$, $y$ | tokens d'entrée / de sortie (dans le vocabulaire $V$) |
| $\Theta$ | poids du modèle |
| $f$ | inférence : $f:\mathbb{Z}_l^n\times\Theta\to\mathbb{Z}_n$ (auto-régressive) |
| $\phi_X$, $\phi_Y$ | obfuscation des **données** (entrée / sortie) |
| $\phi_\Theta$ | obfuscation du **modèle** (les poids) |
| $\tau$ | permutation secrète ($\tau\sim S_n$) |
| $\Pi$ | matrice de permutation de $\tau$ |
| $\hat{P}$, $\hat{Q}$ | matrices **clés** (Alg. 1) : $\hat{P}\cdot\hat{Q}=I$ |
| $Z$ | mapping secret $Z=\{V[i]\to V[\tau[i]]\}$ |

**Modèle de menace (§3.2)** : l'attaquant est l'**opérateur du serveur** — il
possède les poids obfusqués et observe tout ce qui se passe pendant l'inférence
(états cachés, scores d'attention). Il **n'a pas** la permutation $\tau$ ni les
clés. L'objectif d'AloePri : l'inférence sur données obfusquées avec poids
obfusqués donne le **même résultat** que l'inférence en clair (composition
covariante, §4), tout en empêchant l'attaquant de récupérer le texte.

**La chaîne covariante en une phrase** : pour chaque composant, on applique une
transformation **inversible** aux poids ($\phi_\Theta$) qui **annule** la
transformation appliquée aux données ($\phi_X$) — d'où les invariants de la
figure : $\mathrm{KeyMat}\cdot\mathrm{InvKeyMat}=I$,
$\mathrm{Perm}\cdot\mathrm{InvPerm}=I$,
$\mathrm{Scaling}\cdot\mathrm{InvScaling}=I$,
$\mathrm{Mask}\cdot\mathrm{InvMask}=I$. Les blocs 2-6 implémentent chacun de
ces invariants.

> **cellule 6**

## 2. KeyMat / InvKeyMat — Algorithme 1 (§5.2.1)

**La brique $\mathrm{KeyMat}\cdot\mathrm{InvKeyMat}=I$ de la figure.** Le
papier génère des matrices clés $\hat{P} \in \mathbb{R}^{d\times(d+2h)}$ et
inverses $\hat{Q} \in \mathbb{R}^{(d+2h)\times d}$ telles que
$\hat{P}\cdot\hat{Q}=I$ (Algorithme 1, p. 8) :

- $\mathrm{INIT}(d, h, \lambda)$ : tire $B = U + \lambda V$ ($U$
  orthogonale, $V$ gaussienne), $E = E_1 E_2$, $F = F_1 F_2$ (produits
  bas-rang), et $Z$ orthogonale ;
- $\mathrm{KeyMatGen}$ : $\hat{P} = [B\ C\ E]\cdot Z$, où les colonnes de
  $C$ sont dans $\mathrm{null}(F^T)$ ;
- $\mathrm{InvKeyMatGen}$ : $\hat{Q} = Z^T\cdot[B^{-1}\ F\ D]^T$, où les
  lignes de $D$ sont dans $\mathrm{null}(E)$.

Le paramètre $\lambda$ borne la norme de $\hat{P}$ (important en
demi-précision bf16).

**Dans la miniature ci-dessous**, $h = 8$ : les matrices clés sont **actives**
($\hat{P}\in\mathbb{R}^{64\times 80}$, $\hat{Q}\in\mathbb{R}^{80\times 64}$) et la
cellule vérifie $\hat{P}\cdot\hat{Q}=I$. Elles n'agissent qu'à la frontière
`hidden_size` : sur le vrai modèle, le **chaînage inter-couches**
$\hat{P}/\hat{Q}$ ($h=128$, hidden 4352) est appliqué en **section 9.2**
(`transform_chained`).

In [ ]:
# cellule 7
from aloepri.key_matrix import init_key_matrix, key_mat_gen, inv_key_mat_gen
import numpy.random as npr
base = init_key_matrix(d=64, h=8, lam=0.3, rng=npr.default_rng(SEED))
P = key_mat_gen(base); Q = inv_key_mat_gen(base)
err = float(np.abs(P @ Q - np.eye(64)).max())
assert err < 1e-10, f"P̂·Q̂=I attendu, erreur max {err}"
print(f"P̂ ({P.shape}) · Q̂ ({Q.shape}) = I, erreur max {err:.2e} ✓")

> **cellule 8**

## 3. Perm & mapping secret — §5.2.2, §5.3

**Les briques $\mathrm{Perm}\cdot\mathrm{InvPerm}=I$ et `Secret Vocab
Mapping` de la figure.** Le client tire une permutation secrète
$\tau \sim S_n$ (§5.2.2). Elle sert à **deux usages** :

1. **Obfuscation des poids** : la matrice de permutation $\Pi$ (de $\tau$)
   est appliquée aux lignes de l'embedding et de la tête de sortie :
   $\tilde{W}_{\mathrm{embed}} = \Pi \cdot W^{\star}_{\mathrm{embed}}
   \cdot \hat{P}_{\mathrm{embed}}$ et
   $\tilde{W}_{\mathrm{head}} = \hat{Q}_{\mathrm{head}} \cdot
   W^{\star}_{\mathrm{head}} \cdot \Pi^T$ (équations §5.2.2 — $\hat{P}$/
   $\hat{Q}$ étant désactivés ici, il reste $\Pi$ à droite et $\Pi^T$ à
   gauche, qui s'annulent) ;
2. **Mapping secret en ligne (§5.3)** : $Z = \{V[i] \to V[\tau[i]]\}$ — le
   client permute localement les ids de ses tokens avant de les envoyer au
   serveur, et dépermute la réponse. C'est **la protection du texte** :
   l'attaquant ne récupère que des ids permutés, illisibles sans $\tau$.

La cellule construit $\tau$, vérifie $\Pi\cdot\Pi^T = I$, et simule le
round-trip du mapping secret :

In [ ]:
# cellule 9
import numpy as np
V = 1000
rng = np.random.default_rng(SEED)
perm = rng.permutation(V)
unperm = np.empty_like(perm); unperm[perm] = np.arange(V)
# vérification : Π·Π⁻¹ = Id
assert (perm[unperm] == np.arange(V)).all() and (unperm[perm] == np.arange(V)).all()
print(f"Π construite : {V} tokens, inverse exacte ✓")

> **cellule 10**

## 4. Noise / Scaling — §5.2.2, §5.2.4

**La brique $\mathrm{Scaling}\cdot\mathrm{InvScaling}=I$ de la figure.**
Deux usages distincts :

1. **Bruit d'embedding / de tête (§5.2.2)** :
   $W^{\star}_{\mathrm{embed}} = W_{\mathrm{embed}} + \alpha_e \cdot
   E_{\mathrm{embed}}$ avec $E \sim \mathcal{N}(0, \sigma^2 I)$ où
   $\sigma$ est l'écart-type des poids — le rapport bruit/signal vaut
   $\alpha$ par construction. C'est la **seule dégradation non compensée** du
   schéma (coût qualité, mesuré +13-19 % de perplexité) ;
2. **Scaling FFN (§5.2.4)** :
   $\tilde{W}_{\mathrm{up}} = \hat{Q}_{\mathrm{up}} \cdot
   W_{\mathrm{up}} \cdot \hat{H}_{\mathrm{ffn}} \cdot
   \hat{Z}_{\mathrm{ffn}}$ et
   $\tilde{W}_{\mathrm{down}} = \hat{Z}_{\mathrm{ffn}}^{-1} \cdot
   \hat{H}_{\mathrm{ffn}}^{-1} \cdot W_{\mathrm{down}} \cdot
   \hat{P}_{\mathrm{down}}$ — le scaling $\hat{H}_{\mathrm{ffn}}$ est
   **annulé exactement** par son inverse sur `down_proj` (un
   $\mathrm{Scaling}\cdot\mathrm{InvScaling}=I$), et il est placé sur
   `up_proj` (hors de la `SiLU`) pour rester exact.

La cellule vérifie le rapport bruit/signal
($\sigma(\mathrm{bruit})/\sigma(\mathrm{poids}) \approx \alpha_e$) et la
compensation exacte du scaling FFN :

In [ ]:
# cellule 11
w = torch.randn(64, 128)
noise = ALPHA_E * torch.randn_like(w) * w.std()   # rapport bruit/signal = α_e
assert abs(noise.std() / w.std() - ALPHA_E) < 0.1
print(f"σ(bruit)/σ(poids) ≈ {noise.std()/w.std():.2f} ≈ α_e ✓")

In [ ]:
# cellule 12
h = 64
rng = np.random.default_rng(SEED + 7)
neu = rng.permutation(h)
scale = torch.exp(0.1 * torch.randn(h))
assert len(set(neu.tolist())) == h
print(f"FFN : permutation de {h} neurones + scalings ∈ [exp(±0.1·N)] ✓")

> **cellule 13**

## 5. Block & Head Perm, Rot — Algorithme 2 (§5.2.3)

**Les briques `Block & Head Perm`, `Rot`, `RoPE & Softmax` de la figure.** Le
papier obfusque l'attention par deux familles de transformations :

- **Intra-tête (Algorithme 2)** : sur chaque groupe de têtes,
  $\tilde{W}_q = \hat{Q}_q \cdot W_q \cdot \hat{R}_{qk} \cdot
  \hat{H}_{qk} \cdot \hat{Z}_{\mathrm{block}}$ et
  $\tilde{W}_k = \hat{Q}_k \cdot W_k \cdot \hat{R}_{qk} \cdot
  \hat{H}_{qk}^{-1} \cdot \hat{Z}_{\mathrm{block}}^T$ — où
  $\hat{R}_{qk}$ est une rotation 2D par paire RoPE, $\hat{H}_{qk}$ un
  scaling diagonal par paire, et
  $\hat{Z}_{\mathrm{block}} = \mathrm{BlockPerm}(\beta, \gamma, \zeta,
  m_{\mathrm{blocks}})$ une **permutation par fenêtres dynamiques** des blocs
  $2\times 2$ de RoPE (lignes 9-19 de l'Algorithme 2). Les valeurs $v/o$
  portent $\hat{U}_{vo}$ (matrice aléatoire, $\hat{U}_{vo}^{-1}$ sur $o$) ;
- **Inter-têtes** : deux permutations $\tau_{kv} \sim S_{m_{kv}}$ et
  $\tau_{\mathrm{group}} \sim S_{m/m_{kv}}$ mélangent les têtes (K/V au
  niveau tête, Q/O au niveau groupe).

**⚠️ Ce qui est actif sur Qwen3 (après le correctif du 24/08)** :
$\hat{R}$, $\hat{Z}$ et $\hat{H}$ sont **désactivés** — les RMSNorm de tête
`q_norm`/`k_norm` de Qwen3 multiplient par un $\gamma$ appris non constant, et
une rotation/permutation dense de `head_dim` ne commute pas avec cette
multiplication (voir le bloc 6). Restent exacts : les **permutations de têtes**
($\tau_{kv}$, $\tau_{\mathrm{group}}$) et $\hat{U}_{vo}$ (v/o, sans norme).
Le bloc 6 démontre précisément pourquoi.

In [ ]:
# cellule 14
d_head = 32
beta = BETA  # β du papier (8) ; l'exemple ci-dessous illustre β_ex = 3 blocs
# Ẑ : permutation de blocs de largeur d_head//3 (exemple β_ex=3). d_head=32
# n'est pas multiple de 3 → les lignes restantes sont laissées à l'identité
# pour que Ẑ soit une permutation complète (orthogonale).
blk = [1, 2, 0]
w = d_head // len(blk)
Z = torch.zeros(d_head, d_head)
for j, src in enumerate(blk):
    Z[j*w:(j+1)*w, src*w:(src+1)*w] = torch.eye(w)
for r in range(len(blk) * w, d_head):
    Z[r, r] = 1.0
assert torch.allclose(Z @ Z.T, torch.eye(d_head)), "Ẑ doit être une permutation (orthogonale)"
# Û_vo : orthogonale via QR
U, _ = torch.linalg.qr(torch.randn(d_head, d_head))
assert torch.allclose(U.T @ U, torch.eye(d_head), atol=1e-5), "Û_vo doit être orthogonale"
print("R̂/Ẑ/Û_vo : facteurs orthogonaux vérifiés ✓")

> **cellule 15**

> **Correctif Qwen3 (2026-08-24, commit 9f6355e)** — les RMSNorm de tête
> `q_norm`/`k_norm` de Qwen3 portent un $\gamma$ appris **non constant**
> (k_norm : de 0,03 à 96,5 sur Qwen3-0.6B) qui ne commute pas avec les
> rotations/permutations denses de `head_dim` :
> $\gamma \odot (\hat{R}\cdot x) \neq \hat{R}\cdot(\gamma \odot x)$.
> Le round-trip logits était cassé (corr 0,35) et la génération dégénérait.
> Depuis ce correctif, `rope_rotation=False` est automatique sous `q_norm` :
> $\hat{R}$ et $\hat{Z}$ = identité. Restent exacts : permutations de têtes
> ($\tau_{kv}$/$\tau_{\mathrm{group}}$), $\hat{U}_{vo}$ (v/o, aucune
> norme), permutation de vocabulaire, bruit d'embedding, FFN. Compromis : la
> défense d'attention se réduit au mélange de têtes + $\hat{U}_{vo}$ — la
> permutation de vocabulaire reste la protection effective du texte (ids
> permutés illisibles sans la clé).

> **cellule 16**

## 6. RoPE & Softmax, SiLU — les points de non-linéarité

Une transformation covariante doit **commuter** avec les opérations non
linéaires du réseau, sinon le round-trip casse. Trois cas :

1. **RoPE (rotation)** : $\hat{R}$ est une rotation dans les plans RoPE —
   elle **commute avec la rotation RoPE elle-même** (mêmes plans) et préserve
   les normes → elle passe à travers `q_norm` (RMSNorm) *si* le $\gamma$ de
   la norme est constant ;
2. **Softmax / SiLU (élémentwise)** : une **permutation** commute avec toute
   opération appliquée composant par composant
   ($\mathrm{perm}(\mathrm{act}(x)) = \mathrm{act}(\mathrm{perm}(x))$) —
   c'est pourquoi les permutations de têtes et de neurones FFN sont exactes ;
   un **scaling diagonal** ne commute pas avec `SiLU`
   ($\mathrm{silu}(s\cdot z) \neq s\cdot\mathrm{silu}(z)$), d'où le
   placement du scaling FFN sur `up_proj` (hors SiLU) ;
3. **Le $\gamma$ appris de q_norm (le piège Qwen3)** :
   $q\text{norm}(x) = \gamma \odot (x/\mathrm{rms}(x))$. Pour une rotation
   dense $\hat{R}$ : $\gamma \odot (\hat{R}\cdot x) \neq
   \hat{R}\cdot(\gamma \odot x)$ dès que $\gamma$ n'est **pas constant** —
   c'est exactement le défaut qui rendait les vrais Qwen3 inutilisables
   (corr logits 0,35) et qui a motivé `rope_rotation=False`.

La cellule démontre numériquement ce dernier point :

In [ ]:
# cellule 17
# γ⊙(R̂·x) vs R̂·(γ⊙x) — pourquoi le γ appris de q_norm casse les rotations denses
import torch

d_head = 128
torch.manual_seed(0)
# une rotation 2D par paire (i, i+d/2) — comme R̂ dans l'espace "half"
theta = torch.rand(d_head // 2) * 2 * 3.14159
R = torch.zeros(d_head, d_head)
half = d_head // 2
for i in range(half):
    c, s = torch.cos(theta[i]), torch.sin(theta[i])
    R[i, i] = c; R[i, half + i] = -s
    R[half + i, i] = s; R[half + i, half + i] = c

x = torch.randn(d_head)
gamma_const = torch.ones(d_head)                    # cas jouet (γ=1)
gamma_reel = torch.rand(d_head) * 10 + 0.01         # cas Qwen3 (k_norm : 0,03 → 96,5)

err_const = (gamma_const * (R @ x) - R @ (gamma_const * x)).abs().max()
err_reel = (gamma_reel * (R @ x) - R @ (gamma_reel * x)).abs().max()
print(f"γ constant  : erreur de commutation = {err_const:.2e}  → R̂ commute ✓")
print(f"γ réel (non constant) : erreur de commutation = {err_reel:.2e}  → R̂ ne commute PAS ✗")
assert err_const < 1e-5, "avec γ constant, R̂ doit commuter (arrondi près)"
assert err_reel > 0.1, "avec γ non constant, R̂ ne doit PAS commuter"
print("→ conclusion : sur Qwen3 (γ appris non constant), R̂/Ẑ doivent être à l'identité (rope_rotation=False)")

> **cellule 18**

## 7. La chaîne complète — l'obfuscation offline (§5.2.2–5.2.4)

La figure 2 assemble les briques : `Perm` + `Noise` + `KeyMat` sur l'embedding
et la tête, `Block & Head Perm` + `Rot` + `Mask` + `KeyMat` sur l'attention,
`Perm` + `Scaling` + `KeyMat` sur le FFN. L'ordre des opérations sur chaque
poids suit les équations des §5.2.2-5.2.4.

**Preuve locale (miniature)** : on construit un petit Qwen3 aléatoire, on
applique la chaîne complète (avec les briques actives sur Qwen3 : permutation
de vocabulaire, bruit, permutations de têtes, $\hat{U}_{vo}$, FFN —
$\hat{R}/\hat{Z}$ off), et on vérifie que les **logits sont préservés modulo
la permutation de vocabulaire** (le round-trip exact, garantie de la
composition covariante, §4) :

In [ ]:
# cellule 19
from aloepri.check_arch import check  # API réelle : check() (le brief disait check_arch)
import contextlib, io

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    rc = check(MODEL)  # télécharge uniquement la config, jamais les poids
print(buf.getvalue(), end="")
assert rc == 0, "check() a signalé des hypothèses en échec"
assert "toutes les hypothèses sont satisfaites" in buf.getvalue(), \
    "le rapport doit conclure : toutes les hypothèses sont satisfaites"
print("✓ toutes les hypothèses sont satisfaites — la transformation peut être lancée")

In [ ]:
# cellule 20
# Round-trip exact sur un Qwen3 miniature — la chaîne complète (CPU, rapide)
import torch
from transformers.models.qwen3.configuration_qwen3 import Qwen3Config
from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM
from aloepri.model_transform import obfuscate_model_in_place

torch.manual_seed(7)
cfg = Qwen3Config(vocab_size=256, hidden_size=128, intermediate_size=256,
                  num_hidden_layers=2, num_attention_heads=8, num_key_value_heads=4,
                  head_dim=32, max_position_embeddings=64, rope_theta=1e6,
                  tie_word_embeddings=False, bos_token_id=250, eos_token_id=251)
model = Qwen3ForCausalLM(cfg).eval()
ids = torch.tensor([[5, 60, 120, 200, 30, 7]])

with torch.no_grad():
    logits_clair = model(ids).logits                      # baseline (texte clair)

# α_e=α_h=0 : le bruit est la SEULE dégradation non compensée du schéma (§5.2.2) —
# on le retire pour démontrer l'exactitude de la composition covariante (§4).
keys = obfuscate_model_in_place(model, cfg, seed=0, alpha_e=0.0, alpha_h=0.0,
                                beta=1, rope_scaling=None)   # auto → off sous q_norm
ids_perm = torch.tensor([[keys.vocab_permutation[int(t)] for t in ids[0]]])
with torch.no_grad():
    logits_obf = model(ids_perm).logits                   # modèle obfusqué (ids permutés)

# déper mute les logits : logits_obf[..., perm[t]] ≈ logits_clair[..., t]
cols = torch.tensor([keys.vocab_permutation[t] for t in range(cfg.vocab_size)])
rel = (logits_obf[..., cols] - logits_clair).abs().max().item() / logits_clair.abs().max().item()
print(f"round-trip (sans bruit) : erreur relative max = {rel:.2e}")
assert rel < 1e-3, "le round-trip doit être exact (composition covariante)"
print("✓ les logits sont préservés modulo la permutation — l'inférence sur le modèle obfusqué == l'inférence en clair, texte dépermuté côté client")
print("(avec le bruit α_e>0 activé, l'erreur devient ~1e-1 — le coût assumé du §5.2.2)")

> **cellule 21**

#### 7.2 La chaîne complète sur le vrai Qwen3-8B (Modal, `RUN_HEAVY`)

La même chaîne, appliquée cette fois au **vrai Qwen3-8B** (16 Go) en streaming
mémoire-léger dans un conteneur Modal, avec le **chaînage inter-couches**
$\hat{P}/\hat{Q}$ (h>0) et les clés qui restent côté client — c'est l'objet
de la **section 9.2** (`transform_chained`, sur le modèle fine-tuné 9.1b).

> **cellule 22**
## 9. Sécurité — de l'entraînement à l'évaluation

Stratégie : (1) **fine-tuning complet** de Qwen3-8B sur un corpus synthétique
généré avec **DeepSeek + gen_corpus_gepa_codex** (modifier les poids du
modèle) ; (2) **AloePri complet** (h>0, α_e=0.3) ; (3) **VMA complète**
(Table 9) ; (4) **ISA** ; (6) **précision sur frwiki**. IMA : plus tard (9.5).


> **cellule 23**
### 9.1 Génération du corpus avec DeepSeek + gen_corpus_gepa_codex

Le projet https://github.com/mauceri/gen_corpus_gepa_codex génère un corpus de
notes françaises synthétiques via DSPy/**GEPA**, avec **DeepSeek** comme LLM
générateur (`DEEPSEEK_API_KEY` dans l'environnement).

- `promptGenGEPA.py` : optimise un prompt avec GEPA (optimisation génétique de
  prompts DSPy), sauvegarde le meilleur prompt (`GEPAPrompt.txt`) ;
- `generate_corpus_dspy.py` : génère le corpus JSONL — pour chaque triplet
  `(theme, categorie, type_de_document)`, DeepSeek produit une note validée
  (7 clés : `contenu`, `url`, `date`, `expressions_clefs`, `type_de_document`,
  `theme`, `categorie`) ; validation : `contenu` commence par l'étiquette de
  catégorie, `expressions_clefs` (1-8) apparaissent dans `contenu`, URL/date
  plausibles, `theme`/`categorie` identiques aux entrées.

CLI (dans `~/gen_corpus_gepa_codex`) :
- optimiser + générer : `DSPY_CACHEDIR=.dspy_cache python promptGenGEPA.py --count 50 --gepa-prompt GEPAPrompt.txt --output corpus_gepa.jsonl`
- générer avec un prompt existant : `DSPY_CACHEDIR=.dspy_cache python generate_corpus_dspy.py --count 50 --output corpus.jsonl --model deepseek-chat`

Corpus utilisé ici : `~/gen_corpus_gepa_codex/corpus_synth_clean_10000.jsonl`
(10 000 textes, ~1,74 M tokens ; les sorties de génération, ~138 k textes,
sont dans `gepa_llm_calls.log`).


> **cellule 24**

#### 9.1b Full fine-tuning de Qwen3-8B (stratégie 1)

L'objectif : **modifier les poids** du modèle (W_e, attention, FFN, head)
pour que la référence publique de la VMA (Table 9) devienne fausse. Le
fine-tuning **complet** (tous les paramètres) s'appuie sur le corpus
synthétique GEPA (9.1a), français hors distribution généré par DeepSeek.

`finetune_corpus` (modal_app.py) :
1. charge le modèle source en **bf16** (poids + grads + états AdamW bf16)
   et active le **gradient checkpointing** — mémoire ≈ 64,5 Go → A100-80GB
   (AdamW fp32 classique ferait 96,8 Go, infaisable) ;
2. tokenise le corpus GEPA en séquences de `seq_len` tokens ;
3. boucle d'entraînement : AdamW (lr 2e-5) + autocast bf16, loss =
   cross-entropie next-token ;
4. sauvegarde sur le volume `obfuscator-models/qwen3-8b-ft-gepa`.

Budget : A100-80GB, ~30-60 min, ~4-8 $.

In [ ]:
# cellule 25
# 9.1b Full fine-tuning de Qwen3-8B sur le corpus GEPA (TOUS les paramètres)
#
# Objectif : modifier les poids du modèle (stratégie 1) — W_e, attention,
# FFN, head changent → la référence publique de la VMA (Table 9) devient
# fausse. Budget : A100-80GB, bf16 COMPLET (poids 16 Go + grads 16 Go +
# états AdamW bf16 32 Go ≈ 64,5 Go + activations checkpointées) — AdamW
# fp32 classique (m/v 64,6 Go) porterait le total à 96,8 Go, infaisable.
# ~30-60 min, ~4-8 $.
#
# La fonction `finetune_corpus` (modal_app.py) fait exactement ceci :
#   1. charge le modèle source en bf16 (poids + grads + états AdamW) et
#      active le gradient checkpointing (activations re-calculées) ;
#   2. tokenise le corpus GEPA (séquences de `seq_len` tokens) ;
#   3. boucle d'entraînement : AdamW (lr 2e-5) + autocast bf16 sur GPU,
#      loss = cross-entropie next-token ;
#   4. sauvegarde le modèle fine-tuné sur le volume
#      `obfuscator-models/{out_subdir}`.
# Le code de la boucle (extrait commenté de finetune_corpus) :
#   for step in range(epochs * steps_per_epoch):
#       idx = torch.randint(0, n_seq, (batch_size,))
#       with torch.amp.autocast("cuda", dtype=torch.bfloat16):
#           out = model(corpus[idx].cuda(), labels=corpus[idx].cuda())
#       opt.zero_grad(); out.loss.backward(); opt.step()

if RUN_HEAVY:
    !~/modal-venv/bin/modal run modal_app.py::finetune_corpus \
        --model-name Qwen/Qwen3-8B --epochs 5 --batch-size 8 \
        --seq-len 128 --lr 2e-5 --out-subdir qwen3-8b-ft-gepa
else:
    print("[RUN_HEAVY=False] fine-tuning 8B sauté — code et budget ci-dessus ; "
          "résultat attendu : qwen3-8b-ft-gepa sur le volume")

> **cellule 26**

#### 9.2 AloePri COMPLET — h>0, matrices clés (§5.4)

AloePri h>0 reconstruit le modèle avec `hidden_size = d + 2h` (h=128 →
4352) et applique le **chaînage global P̂/Q̂** : les clés n'agissent qu'à la
frontière `hidden_size` et s'annulent par chaînage entre couches.

`transform_chained` (modal_app.py) applique, sur le modèle **fine-tuné**
(9.1b — pas la base) :
- `embed·P̂` ; `q/k/v/gate/up·Q̂ᵀ` (Wnorm fusionnée) ; `o/down·P̂ᵀ` ; `head·Q̂ᵀ` ;
- les normes → κ (§5.2.5, κ empirique par couche) ;
- deux corrections de l'Algorithme 1 du papier : F1/F2 ~ N(0,1/d) ;
  scaling C √(h/d) ;
- bruit d'embedding **α_e** — la seule défense directe contre la VMA produit
  (Table 9). **Mesuré (2026-09-01, attaque corrigée)** : α_e=0,3 → 90,8 % de
  récupération (insuffisant) ; α_e=**1,0** → 8,35 % (conforme au papier) —
  c'est le réglage défensif retenu.

Budget : CPU, ~1-1,5 h, ~1-2 $. Sortie : `qwen3-8b-ft-h128-a1-h02`.

In [ ]:
# cellule 27
# 9.2 AloePri COMPLET (h>0, matrices clés) sur le 8B fine-tuné — α_e=1.0
#
# `transform_chained` (modal_app.py) reconstruit le modèle avec
# hidden_size = d + 2h (h=128 → 4352) et applique le chaînage global P̂/Q̂ :
#   embed·P̂ ; q/k/v/gate/up·Q̂ᵀ (Wnorm fusionnée) ; o/down·P̂ᵀ ; head·Q̂ᵀ ;
# normes → κ (§5.2.5, κ empirique par couche). Deux corrections de
# l'Algorithme 1 du papier : F1/F2 ~ N(0,1/d) ; scaling C √(h/d).
# α_e=1.0 : bruit d'embedding relatif à σ(W) — le réglage défensif MESURÉ
# (8,35 % VMA, conforme au papier ; α_e=0,3 laissait 90,8 %).
# Budget : CPU, ~1-1,5 h, ~1-2 $. Sortie : `qwen3-8b-ft-h128-a1-h02`.

if RUN_HEAVY:
    # on obfusque le modèle FINE-TUNÉ (9.1b) depuis le volume — pas la base.
    # α_e=1,0 : le réglage défensif MESURÉ (8,35 % VMA, conforme au papier) —
    # α_e=0,3 laissait 90,8 % de récupération.
    !~/modal-venv/bin/modal run modal_app.py::transform_chained \
        --seed 0 --alpha-e 1 --alpha-h 0.2 --h 128 \
        --model-name /models/qwen3-8b-ft-gepa --out-subdir qwen3-8b-ft-h128-a1-h02
else:
    print("[RUN_HEAVY=False] transform_chained sauté — code et budget ci-dessus ; "
          "résultat attendu : qwen3-8b-ft-h128-a1-h02 (hidden 4352, α_e=1,0)")

> **cellule 28**

#### 9.3 VMA COMPLÈTE (Table 9) — profondeur maximale

L'attaque VMA produit cherche à retrouver la permutation Π en comparant des
produits de poids obfusqués à la référence publique. Ici, profondeur
maximale : **les 36 couches** + vote complet.

`vma_product_full` (modal_app.py), pour chaque vue (gate, up) × couche :
1. produit `W̃_e·W̃_gateᵀ` (ou up) — les P̂/Q̂ s'annulent par chaînage ;
2. RowSort chunké (élimine Ẑ_ffn) ;
3. appariement NN sur `subset_size` tokens ;
4. **vote majoritaire** sur les 36 couches (par vue) puis vote global.

Limites : W_e·W_h et gram q·k (vues V×V = 46 Go) dépassent l'A100-40GB ;
la vue **up est inerte par construction** (Ŝ_ffn sur up_proj, non
éliminé par RowSort). Précision : produits en **fp32** (le chaînage
P̂Q̂≈I en bf16 casse l'appariement — mesuré) et agrégation par **somme
des similarités** sur les couches (le vote par mode détruisait le
signal). Budget : A100-40GB, ~1-1,5 h, ~2-4 $.

In [ ]:
# cellule 29
# 9.3 VMA COMPLÈTE (Table 9) — profondeur maximale : toutes les couches + vote
#
# `vma_product_full` (modal_app.py) attaque le modèle h>0 contre la référence
# publique : pour chaque vue × chaque couche, produit W̃_e·W̃_gateᵀ (ou up) —
# les P̂/Q̂ s'annulent par chaînage — RowSort (élimine Ẑ_ffn), appariement NN
# sur 2000 tokens, puis VOTE majoritaire sur les 36 couches (par vue) et
# vote global entre vues. Amélioration de profondeur : 36/36 couches (au lieu
# d'un sous-ensemble), vote complet (mécanisme du papier).
# W_e·W_h : ÉCARTÉ pour l'instant (vue V×V, 46 Go — à revoir quand le reste
# sera au point). Vues V×V (gram q·k) : limite A100-40GB documentée.
# Budget : A100-40GB, ~1-1,5 h, ~2-4 $.

if RUN_HEAVY:
    !~/modal-venv/bin/modal run modal_app.py::vma_product_full \
        --model-subdir qwen3-8b-ft-h128-a1-h02 --subset-size 2000 --views gate
else:
    print("[RUN_HEAVY=False] vma_product_full sauté — code et budget ci-dessus ; "
          "résultat attendu : TTRSR par couche + votes, puis un bloc "
          "RÉSUMÉ lisible (interprétation incluse) ; rapport "
          "artifacts/vma_produit_8b_complet.md)")

> **cellule 30**

#### 9.4 ISA — Internal State Attack (canal hidden)

L'attaquant (= opérateur serveur) capture l'état caché d'une couche profonde
(layer 18) sur le prompt secret (ids permutés) et cherche à retrouver
l'entrée. Deux variantes :

- **Par gradient (soft tokens + recuit)** : `isa_attack` — l'optimisation
  converge (loss → 0,007) mais récupère **0 %** : des soft tokens de
  l'enveloppe convexe du simplexe reproduisent l'état caché sans être le
  prompt → **ne permet pas de conclure** (artefact de la relaxation
  continue, identifié par la revue) ;
- **Discrète (vocab-matching k-way)** : `isa_vocab_attack` — à chaque
  position, le vrai token est mélangé à k−1 leurres et le canal doit
  l'identifier. **Résultat mesuré : 100 %** (10/10, k=64, baseline 1/k) —
  le canal `hidden` est **INFORMATIF** sur h>0, même à α_e=1,0.

Les ids récupérés sont ceux du **modèle** (permutés) — sans la clé Π
(côté client), aucun texte : **la confidentialité repose uniquement sur la
permutation**. La cellule envoie les ids permutés du prompt secret (clé
seed 0) — jamais d'ids clairs au serveur.
Budget : A100-40GB, ~30-60 min, ~1-2 $.

In [ ]:
# cellule 31
# 9.4 ISA — variante DISCRÈTE (vocab-matching k-way) — la mesure qui tranche
#
# La variante par gradient (`isa_attack`) converge mais récupère 0 % :
# artefact de la relaxation continue (l'enveloppe convexe du simplexe est
# plus grande que les sommets) — elle ne prouve rien sur le canal. La
# recherche DISCRÈTE mélange le vrai token à k−1 leurres et mesure si le
# canal l'identifie : 100 % = informatif, ~1/k = sous-déterminé.
# Mesuré (2026-09-01) : 100 % sur α_e=1,0 — canal hidden INFORMATIF.
#
# `isa_vocab_attack` (modal_app.py) fait exactement ceci :
#   1. capture l'état caché de la couche `layer` sur le prompt (ids permutés) ;
#   2. pour chaque position : préfixe (teacher-forced ou greedy) + k−1
#      leurres + le vrai token, forward batch, distance MSE/cos ;
#   3. argmin → le canal a-t-il identifié le vrai token ? (taux vs 1/k).
# Budget : A100-40GB, ~30-60 min, ~1-2 $.

if RUN_HEAVY:
    # ids = les ids PERMUTÉS du prompt secret (calculés côté client avec la
    # clé seed 0 — jamais envoyés en clair au serveur)
    import json as _json
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL)
    _perm = _json.load(open("artifacts/obfuscation_keys.json"))["vocab_permutation"]
    _secret = "Quelle est la capitale de la France ?"
    _ids_perm = ",".join(str(_perm[str(i)]) for i in tok.encode(_secret))
    !~/modal-venv/bin/modal run modal_app.py::isa_vocab_attack --ids $_ids_perm         --layer 18 --k 64 --channel hidden --teacher-forcing --metric mse         --model-ref qwen3-8b-ft-h128-a1-h02
else:
    print("[RUN_HEAVY=False] ISA discret sauté — code et budget ci-dessus ; "
          "attendu : taux ~100 % = canal informatif (mesuré 2026-09-01)")

> **cellule 32**
### 9.5 IMA (Inversion Model Attack) — réservé

Attaque par entraînement d'un modèle d'inversion (appendice D.1 du papier).
À traiter plus tard — dépend de la défense de Π (VMA), qui est le point 9.3.

In [ ]:
# cellule 33
# 9.6 Précision sur Wikipedia français (échantillon NDJSON embarqué)
#
# Métriques : PERPLEXITÉ + précision NEXT-TOKEN (top-1) sur un échantillon
# de textes frwiki. Comparaison : base (Qwen3-8B) vs fine-tuné (9.1b) vs
# fine-tuné+obfusqué h>0 (9.2). L'obfusqué reçoit les ids PERMUTÉS et ses
# logits sont dépermutés pour la comparaison.
#
# IMPORTANT (corrigé le 2026-09-01) : le corpus frwiki est du NDJSON
# (fichiers `wiki_XX` sans extension, une ligne = {"id","title","text"}),
# PAS des .txt — l'ancienne version lisait ~/corpus_fr/frwiki et trouvait
# 0 fichiers → « 0 tokens ». Le corpus complet (1,1 Go) n'est pas monté
# dans le conteneur Modal : on embarque un sous-ensemble déterministe
# (wiki_00..09, ~10 Mo) dans l'image via `add_local_dir(/frwiki_sample)`.
#
# La fonction `precision_frwiki` (modal_app.py) fait exactement ceci :
#   1. lit `/frwiki_sample` (NDJSON), tokenise avec Qwen/Qwen3-8B ;
#   2. charge la base depuis HF (réf. publique) + ft + ft+h>0 depuis le
#      volume `obfuscator-models` ;
#   3. pour l'obfusqué : ids PERMUTÉS (perm régénérée par seed 0, comme
#      transform_chained) et logits dépermutés avant comparaison ;
#   4. perplexité + top-1 pour les 3 modèles, résumé lisible en fin de
#      sortie.
# Budget : A100-40GB, ~20-40 min (3 modèles chargés), ~1-2 $.

if RUN_HEAVY:
    !~/modal-venv/bin/modal run modal_app.py::precision_frwiki \
        --seed 0 --n-files 4 --max-tokens 4000 \
        --obf-ref qwen3-8b-ft-h128-a1-h02
else:
    print("[RUN_HEAVY=False] précision frwiki sauté — code et budget ci-dessus ; "
          "attendu : perplexité + top-1 pour base / FT / FT+h>0")


In [ ]:
# cellule 34
# 9.6b Précision sur QUESTIONS françaises (PiaF, dataset `AgentPublic/piaf`)
#
# Complément de 9.6 : le texte courant (frwiki) n'est pas le seul registre —
# les QUESTIONS (texte interrogatif court) sont évaluées ici. `precision_piaf`
# (modal_app.py) télécharge PiaF depuis le Hub HF, concatène `n_questions`
# questions (séparées par eos) jusqu'à `max_tokens`, puis mesure
# PERPLEXITÉ + top-1 next-token sur base (HF) / fine-tuné (9.1b) /
# fine-tuné+obfusqué h>0 (9.2, α_e via `obf_ref`).
# Budget : A100-40GB, ~20-40 min (3 modèles chargés), ~1-2 $.

if RUN_HEAVY:
    !~/modal-venv/bin/modal run modal_app.py::precision_piaf \
        --seed 0 --n-questions 300 --max-tokens 4000 \
        --obf-ref qwen3-8b-ft-h128-a1-h02
else:
    print("[RUN_HEAVY=False] précision PiaF sauté — code et budget ci-dessus ; "
          "attendu : perp. 6,09 / 6,97 / 7,78 (mesuré 2026-09-02, journal 10.5b)")

> **cellule 35**

## 10. Résultats des runs — 2026-09-01 (Qwen3-8B, grandeur nature)

Journal des mesures reconstitué à partir des logs Modal (les sorties Jupyter
ont été perdues lors d'un rechargement — les valeurs ci-dessous proviennent
des logs des apps `modal app logs <app_id>`).

> **⚠️ Correctif 2026-09-01 (revue de code)** : le « VMA = 0,0 % » initial était
> un **artefact de l'attaque bf16 cassée** (produits calculés en bf16, le
> chaînage P̂Q̂≈I y accumule une erreur max|Y−X| ≈ 0,33 qui détruit
> l'appariement même sans défense). Après correction (produits fp32, vue
> gate seule, agrégation par somme des similarités), l'attaque est validée à
> **99,95 %** sur base sans bruit, et la vraie courbe de sensibilité est
> mesurée ci-dessous.

### 10.1 9.1b — Full fine-tuning Qwen3-8B (cellule 25, app `ap-ehynOVo8nI2DQLAnDlUotS`)

- corpus GEPA : 10 000 textes, 1 736 135 tokens → 13 563 séquences de 128 ;
- entraînement : 8475 pas (5 époques), batch 8, lr 2e-5, A100-80GB, bf16
  complet + gradient checkpointing ;
- **loss début = 1,7632 → fin = 0,2725** (min observé 0,1033), **83,1 min** ;
- sortie sur volume : `obfuscator-models/qwen3-8b-ft-gepa`.

### 10.2 9.2 — AloePri complet h>0 (cellule 27, app `ap-xWKzIuoSS2YF2TavJLyvu0`)

- transform_chained sur le modèle **fine-tuné** : seed 0, α_e=0,3, α_h=0,2,
  h=128, κ empirique par couche ;
- **hidden_size = 4352** (d=4096 + 2h=256) — confirmé par le `config.json`
  du volume ; 36 couches, intermediate 12288 ;
- sortie sur volume : `obfuscator-models/qwen3-8b-ft-h128`.

### 10.3 9.3 — VMA produit complète (Table 9), attaque CORRIGÉE

Attaque corrigée (produits **fp32**, vue **gate** seule, **somme des
similarités** sur les 36 couches, z-score par ligne, argmax global) — le
contrôle positif sur base sans bruit (`qwen3-8b-base-h128-a0`) :
**99,95 %** → le chemin de code est sain.

| Modèle (FT + h>0 sauf contrôle) | α_e | α_h | TTRSR | app |
|---|---|---|---|---|
| base h>0 (contrôle, sans bruit) | 0 | 0 | **99,95 %** | `ap-hFgHA4IBZy1Xwajr2AKf32` |
| FT + h>0 | 0,01 | 0 | **99,95 %** | `ap-SrPzSm0P1v41vZqYTifC8i` |
| FT + h>0 | 0,3 | 0,2 | **90,8 %** | `ap-qH10woNC4EkkSecVLU3coF` |
| FT + h>0 | **1,0** | **0,2** | **8,35 %** | `ap-qtR6Miib9bE7tZKLt7Twzi` |

Lecture : la défense contre la VMA produit est le **bruit d'embedding α_e**
(le papier recommande α_e=1,0 — notre mesure 8,35 % est cohérente avec leurs
13-25 %). À α_e=0,3 (notre ancien réglage) la VMA récupère encore 90,8 % :
**α_e=0,3 est insuffisant**. Le fine-tuning n'y contribue pas (la courbe de
la base sans bruit est identique) — l'affinage ne protège pas la VMA produit.

### 10.4 9.4 — ISA (canal hidden, couche 18)

- **Variante gradient** (app `ap-hHHuhEwV7DpI07NJbtZWpM`, modèle α_e=0,3) :
  400 steps + phase 2, recuit τ 3,0→0,15 ; loss 0,934 → 0,0070 (converge) ;
  récupération **0,0 %** — mais **artefact de la relaxation continue** (des
  soft tokens de l'enveloppe convexe reproduisent l'état caché sans être le
  prompt) : ne permet PAS de conclure ;
- **Variante discrète (vocab-matching k-way)** (app `ap-7UqTvY4sf8bDVRIteoyvqM`,
  modèle **α_e=1,0** — le réglage le plus défensif) : à chaque position, le
  vrai token est mélangé à k−1=63 leurres, recherche discrète, teacher-forced,
  MSE → **taux d'identification = 100 %** (10/10) ;
- **Conclusion (revue confirmée)** : le canal hidden à la couche 18 est
  **INFORMATIF** — l'attaquant (opérateur serveur) récupère les ids PERMUTÉS
  de l'entrée. La protection du texte ne repose que sur la **clé Π** côté
  client (sans elle, pas de dépermutation) ; AloePri h>0 ne protège pas les
  états cachés.

### 10.5 9.6 — Précision sur Wikipedia français

- échantillon : 4 fichiers NDJSON (`/frwiki_sample`), **4000 tokens**, seed 0 ;
- **à α_e=0,3** (app `ap-EfbZ1CNerNNvXwLT2iu4pY`) :

| Modèle | Perplexité | Top-1 next-token |
| :--- | ---: | ---: |
| base (Qwen3-8B) | 1,88 | 0,8092 |
| fine-tuné (9.1b) | 2,01 | 0,7887 |
| fine-tuné + obfusqué h>0 (α_e=0,3) | 2,14 | 0,7742 |

- **à α_e=1,0** (app `ap-af9FuJLIG5TskzbVU73KVv`) :

| Modèle | Perplexité | Top-1 next-token |
| :--- | ---: | ---: |
| base (Qwen3-8B) | 1,88 | 0,8092 |
| fine-tuné (9.1b) | 2,01 | 0,7887 |
| fine-tuné + obfusqué h>0 (α_e=1,0) | **2,28** | **0,7527** |

  Coût du réglage défensif recommandé : perplexité 2,01 → 2,28 (+13 % par
  rapport au FT, +21 % vs base), top-1 −3,6 pts vs FT — cohérent avec la
  promesse du papier (« < 3 % de perte » en accuracy sur benchmarks), mais
  non négligeable sur ce petit échantillon.
- Réserve : 4000 tokens avec top-1 0,809 anormalement haut (balisage
  répétitif) — échantillon trop petit pour être cité ; à élargir.

### 10.5b Précision sur QUESTIONS françaises (PiaF)

- dataset `AgentPublic/piaf` (3835 Q/R, Hub HF), questions SEULES (texte
  interrogatif court), 300 questions → 4000 tokens, app
  `ap-eqeprYH1JGfumxgUegNIX4` (le run 64 tokens `ap-FXeN2349EBzyFh6wLpC8Hq`
  était trop petit, non retenu) :

| Modèle | Perplexité | Top-1 next-token |
| :--- | ---: | ---: |
| base (Qwen3-8B) | 6,09 | 0,5149 |
| fine-tuné (9.1b) | 6,97 | 0,5029 |
| fine-tuné + obfusqué h>0 (α_e=1,0) | 7,78 | 0,4869 |

Lecture : les questions sont plus dures que le texte courant (perp. base
6,1 vs 1,88 sur frwiki — texte interrogatif moins prédictible). Le FT GEPA
dégrade légèrement (6,1 → 7,0, top-1 −1,2 pt) ; AloePri α_e=1,0 ajoute un
coût net (7,0 → 7,8, top-1 −1,6 pt vs FT ; écart base vs FT+obf : +1,7 de
perplexité, −2,8 pts de top-1). Le coût de la défense sur questions est
plus visible que sur texte courant (frwiki α_e=1,0 : +13 % ; ici +12 % vs
FT — comparable), et le top-1 de l'obfusqué (0,487) reste très au-dessus
du hasard (1/V ≈ 0).

### 10.5c Évaluation Q&A par jugement DeepSeek (PiaF)

Pipeline (02/09) : 150 couples (question, contexte) de PiaF → génération
greedy par la **base** et l'**obfusqué** (α_e=1,0, ids permutés → réponse
dépermutée côté client) → **DeepSeek** (`deepseek-chat`, secret Modal
`deepseek-api-key`) note chaque réponse de 1 à 5 vs la référence PiaF
(prompt calibré : exacte=5, partielle=2, hors sujet=1) → note globale.
App `ap-Euzzink3yZcTUbmRKWjOaA` ; détail sur volume `piaf_eval.json`.

| Modèle | Note globale (1-5) |
| :--- | ---: |
| base (Qwen3-8B) | **4,62** |
| obfusqué (FT + h>0, α_e=1,0) | **4,42** |

Écart **−0,20 pt (−4,3 %)** — 150/150 notés par modèle. Lecture : la
défense α_e=1,0 reste très fonctionnelle en Q&A (4,42/5) ; son coût est
modeste mais réel (−4,3 % vs base), cohérent avec la perplexité/top-1
(frwiki +13 %, PiaF +12 %). La génération dépermutée valide l'usage réel
du modèle obfusqué (interroger = ids permutés, réponse lisible côté client).

### 10.6 Bilan (corrigé)

- **VMA produit** : la défense est le **bruit α_e** (8,35 % à α_e=1,0,
  conforme au papier) ; **α_e=0,3 ne suffit pas (90,8 %)** ; le « 0,0 % »
  initial était un artefact bf16 ;
- **ISA** : le canal hidden est **informatif** (100 % en recherche discrète) —
  la protection du texte repose uniquement sur la clé Π côté client ;
- **Précision** : frwiki (texte courant) ET PiaF (questions) mesurés à α_e=1,0 ;
- **IMA** (9.5) : réservé.

Reste à faire : ISA discret (point 4), précision α_e=1,0 (en cours), élargir
l'échantillon frwiki, et revue de code (docs/REVIEW.md + REVIEW-FIXES.md).


> **cellule 36**

## 11. Annexe — les méthodes de `modal_app.py`

Inventaire des fonctions Modal. Exécution : `!~/modal-venv/bin/modal run
modal_app.py::<fonction> --<args>` (cellules lourdes conditionnées par
`RUN_HEAVY`). Conventions : les clés ne sont jamais sur Modal (volume
`obfuscator-keys` supprimé — posture) ; les modèles vivent sur le volume
`obfuscator-models`.

### Transformation (offline)

| Fonction | Rôle | GPU / durée |
| :--- | :--- | :--- |
| `transform` | obfuscation h=0 (permutation vocabulaire + bruit α_e + attention/FFN) ; écrit modèle + clés sur les volumes | CPU ~30-60 min |
| `transform_chained` | **AloePri h>0** : reconstruction hidden d+2h + chaînage P̂/Q̂ global, κ empirique par couche | CPU ~1-1,5 h |
| `verify` | vérification bit-à-bit (échantillons de lignes/couches) après `transform` | CPU |
| `verify_chained` | idem pour `transform_chained` (tolérance bf16) | CPU |

### Service (online)

| `serve` | service web fail-closed (Bearer, `~/.aloepri-api-key`) ; génération sur ids permutés — sans tokenizer ni clé | L4 |
| `serve_env` | diagnostic d'environnement du service | — |

### Attaques

| Fonction | Rôle | GPU / durée |
| :--- | :--- | :--- |
| `isa_attack` | ISA canal hidden/attn : inversion par descente de gradient (soft tokens + recuit de température) | A100-40GB ~30 min |
| `attention_inversion` | clean-space attack (inversion des scores d'attention) | A100-40GB |
| `vma_attack` | VMA directe (embedding vs table claire, h=0) | A100-40GB |
| `vma_product_attack` | VMA produit (Table 9) sur un sous-ensemble de couches | A100-40GB |
| `vma_product_full` | VMA produit **complète** : 2 vues × 36 couches + vote majoritaire, bloc RÉSUMÉ en fin de sortie | A100-40GB ~1-1,5 h |

### Fine-tuning et précision

| Fonction | Rôle | GPU / durée |
| :--- | :--- | :--- |
| `finetune_corpus` | full FT 8B (tous les paramètres) sur corpus GEPA — bf16 complet + gradient checkpointing | A100-80GB |
| `precision_frwiki` | perplexité + top-1 next-token sur échantillon frwiki (NDJSON embarqué `/frwiki_sample`) : base vs FT vs FT+obfusqué | A100-40GB ~20-40 min |

### Divers

| Fonction | Rôle |
| :--- | :--- |
| `diag` | diagnostic volumes / images |
| `compare_poc` | comparaison POC h=0 vs base (référence interne) |

> **cellule 37**

## 12. Démonstration — que voit l'attaquant après l'attaque VMA (α_e=1,0) ?

À α_e=1,0 (le réglage défensif retenu, cf. 9.2), la VMA produit ne récupère
que **~8,35 % des tokens** (mesuré le 2026-09-01, conforme au papier). Cette
section matérialise ce que cela signifie pour un texte réel : le paragraphe
ci-dessous (article Wikipédia) est tokenisé, chaque token est marqué
« récupéré » avec probabilité égale au taux mesuré (tirage déterministe,
seed 6 → 8,37 %), puis le texte est reconstruit **mot par mot** : un mot est
affiché s'il contient au moins un token récupéré (l'attaquant en devine une
bribe) ; sinon il est masqué — par `INC` ou par `_` (deux variantes).

**Texte source** — article « Filtre passe-bas »
([Wikipédia](https://fr.wikipedia.org/wiki/Filtre_passe-bas)) :

> Un filtre passe-bas est un filtre qui laisse passer les basses fréquences
> et qui atténue les hautes fréquences, c'est-à-dire les fréquences
> supérieures à la fréquence de coupure. Il pourrait également être appelé
> filtre coupe-haut. Le filtre passe-bas est l'inverse du filtre passe-haut
> et ces deux filtres combinés forment un filtre passe-bande. Le concept de
> filtre passe-bas est une transformation mathématique appliquée à des
> données. Le filtrage passe-bas peut se faire numériquement ou avec des
> composants électroniques. Cette transformation a pour fonction d'atténuer
> les fréquences supérieures à sa fréquence de coupure et ce, dans le but de
> conserver uniquement les basses fréquences. La fréquence de coupure du
> filtre est la fréquence séparant les deux modes de fonctionnement idéaux
> du filtre.

In [ ]:
# cellule 38
# 12.1 Conversion : masque ~8,35 % (taux VMA α_e=1,0) → texte vu par l'attaquant
#
# Principe : l'attaquant a récupéré Π pour ~8,35 % des tokens (taux mesuré
# vma_product_full sur qwen3-8b-ft-h128-a1-h02, 2026-09-01). On matérialise
# le résultat sur un texte réel :
#   1. tokeniser le paragraphe (tokenizer Qwen3-8B, offsets) ;
#   2. marquer chaque token « récupéré » avec probabilité RATE (tirage
#      déterministe seed 6 → 18/215 = 8,37 %) ;
#   3. reconstruire mot par mot : un mot est affiché s'il contient au moins
#      un token récupéré (bribe devinée) ; sinon masqué — variante INC ou _.
# La cellule 12.2 (markdown) montre le résultat FIGÉ — pas besoin de lancer
# celle-ci pour le voir.

import random, textwrap
from transformers import AutoTokenizer

RATE = 0.0835          # taux VMA α_e=1,0 (vote_gate, 2026-09-01)
SEED = 6               # tirage déterministe → 18/215 tokens = 8,37 %
TEXTE = (
    "Un filtre passe-bas est un filtre qui laisse passer les basses "
    "fréquences et qui atténue les hautes fréquences, c'est-à-dire les "
    "fréquences supérieures à la fréquence de coupure. Il pourrait "
    "également être appelé filtre coupe-haut. Le filtre passe-bas est "
    "l'inverse du filtre passe-haut et ces deux filtres combinés forment "
    "un filtre passe-bande. Le concept de filtre passe-bas est une "
    "transformation mathématique appliquée à des données. Le filtrage "
    "passe-bas peut se faire numériquement ou avec des composants "
    "électroniques. Cette transformation a pour fonction d'atténuer les "
    "fréquences supérieures à sa fréquence de coupure et ce, dans le but "
    "de conserver uniquement les basses fréquences. La fréquence de "
    "coupure du filtre est la fréquence séparant les deux modes de "
    "fonctionnement idéaux du filtre."
)

tok = AutoTokenizer.from_pretrained(MODEL)
enc = tok(TEXTE, add_special_tokens=False, return_offsets_mapping=True)
ids, offs = enc["input_ids"], enc["offset_mapping"]
rng = random.Random(SEED)
recognized = [rng.random() < RATE for _ in ids]

def texte_attaquant(placeholder):
    """Reconstruit le texte vu par l'attaquant : mot affiché s'il contient au
    moins un token récupéré, sinon remplacé par `placeholder`."""
    parts, i = [], 0
    while i < len(TEXTE):
        if TEXTE[i] == " ":
            i += 1
            continue
        j = i
        while j < len(TEXTE) and TEXTE[j] != " ":
            j += 1
        mot = TEXTE[i:j]
        cov = [rec for (a, b), rec in zip(offs, recognized) if b > i and a < j]
        parts.append(mot if (any(cov) if cov else False) else placeholder)
        i = j
    return " ".join(parts)

n_tok, n_rec = len(ids), sum(recognized)
vue_inc = texte_attaquant("INC")
vue_und = texte_attaquant("_")
mots = vue_und.split()
n_vis = sum(1 for w in mots if w != "_")
print(f"tokens : {n_tok} | récupérés : {n_rec} ({n_rec/n_tok:.2%})")
print(f"mots   : {len(mots)} | visibles : {n_vis} ({n_vis/len(mots):.1%})")
print()
print("=== variante INC ===")
print(vue_inc)
print()
print("=== variante '_' ===")
print(vue_und)

> **cellule 39**

#### 12.2 Résultat (figé — 8,37 % des tokens récupérés, seed 6)

Le paragraphe vu par l'attaquant après l'attaque VMA à α_e=1,0 (les mots en
**gras** contiennent au moins un token récupéré — l'attaquant en devine une
bribe ; les autres sont masqués) :

**Variante `INC`** :

> INC INC **passe-bas** INC INC INC INC INC INC INC INC INC INC INC INC INC INC INC INC INC INC INC **à**
> INC INC INC INC INC INC INC INC INC INC **coupe-haut.** INC INC INC INC **l'inverse** INC INC INC INC
> INC INC INC INC INC INC INC **passe-bande.** INC INC INC INC INC INC INC INC INC INC INC INC INC INC INC
> INC **peut** INC **faire** **numériquement** INC INC INC **composants** INC INC INC INC INC INC INC INC
> INC INC INC INC INC INC INC INC INC INC INC INC INC INC **uniquement** INC INC INC INC **fréquence** INC
> INC INC **filtre** INC **la** INC **séparant** INC INC INC INC **fonctionnement** **idéaux** INC INC

**Variante `_`** (plus lisible) :

> _ _ **passe-bas** _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ **à** _ _ _ _ _ _ _ _ _ _ **coupe-haut.** _ _ _
> _ **l'inverse** _ _ _ _ _ _ _ _ _ _ _ **passe-bande.** _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ **peut** _
> **faire** **numériquement** _ _ _ **composants** _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _
> **uniquement** _ _ _ _ **fréquence** _ _ _ **filtre** _ **la** _ **séparant** _ _ _ _ **fonctionnement**
> **idéaux** _ _

**Lecture** : 18 tokens sur 215 (8,37 %) récupérés → 16 mots sur 119 (13,4 %)
émergent comme bribes isolées (« passe-bas », « coupe-haut », « l'inverse »,
« composants », « fréquence », « filtre », « la »…). Le texte est
**largement illisible** — l'attaquant devine des fragments, pas le sens.
C'est l'effet de α_e=1,0 : la confidentialité du contenu tient, au prix
d'une qualité dégradée (9.6 : perplexité 2,28). Résultat **figé ici** —
la cellule 12.1 (code) permet de le reproduire avec un autre seed ou texte.

> **cellule 40**

## 13. Analyse de fréquence — où tombent les tokens récupérés par la VMA ?

La VMA à α_e=1,0 récupère 8,35 % des tokens testés. **Mais lesquels ?** Un
token **fréquent** (rang 1 = `,`, 3,4 % du corpus) porte la structure ; un
token **rare** porte le sens spécifique. Cette section mesure si la
récupération est concentrée sur les fréquents (fuite structurelle) ou
répartie.

Méthode :
1. **Distribution de fréquence** : les tokens du corpus GEPA (10 000 textes,
   1,73 M tokens, 7158 types) sont triés par fréquence décroissante →
   diagramme de **Zipf** (rang × fréquence) ;
2. **Échantillonnage ciblé de la VMA** : l'échantillon uniforme (2000 tokens
   sur V=151 643) ne touchait que ~94 tokens GEPA — inexploitable. On teste
   désormais un échantillon **stratifié GEPA** (`sample=gepa-strat`) : les
   100 tokens les plus fréquents + un tirage du reste → chaque classe
   décimale de rang est peuplée ;
3. **Graphe en barres** : classes décimales de rang (1-10, 11-100,
   101-1000, 1001-10000, 10001+), largeur ∝ taille de classe, hauteur =
   % de tokens récupérés dans la classe.

Les deux images sont affichées ci-dessous (générées depuis les ids récupérés
du run VMA — voir le journal cellule 35).

> **cellule 41**

#### 13.1 Les deux figures

**Figure 1 — Distribution de Zipf du corpus GEPA** (rang × fréquence) :

![Zipf corpus GEPA](zipf_gepa.png)

**Figure 2 — Où tombent les tokens récupérés par la VMA** (classes décimales
de rang, largeur ∝ taille de classe, hauteur = % récupéré) :

![Classes de récupération VMA](classes_recup.png)

**Lecture** (run `ap-7pApK1Q9mOc4TAlKDLn2dG`, échantillon stratifié GEPA,
2000 tokens, α_e=1,0) :

| Classe (rang GEPA) | Taille | Récupérés | % |
|---|---|---|---|
| 1-10 (top fréquence) | 10 | 0 | 0,0 % |
| 11-100 | 90 | 9 | **10,0 %** |
| 101-1000 | 900 | 23 | 2,6 % |
| 1001-10000 | 6158 | 172 | 2,8 % |

- Taux global sur l'échantillon stratifié : **10,2 %** (vs 8,35 % en
  uniforme) — la VMA récupère mieux les tokens fréquents du corpus ;
- **Classe 11-100 sur-représentée** (10 %) : ce sont des fragments
  grammaticaux très fréquents — ` que`, ` a`, ` Les`, `ent`, `és`, `is`…
  → **fuite structurelle** : l'attaquant devine la charpente grammaticale
  (conjonctions, auxiliaires, terminaisons), pas le sens spécifique ;
- **Top-10 non récupéré** (0 %) : ` ,`, ` de`, ` .`, ` la` — probablement
  trop de collisions entre ces tokens ultra-fréquents dans le RowSort ;
- Les tokens **rares** (101-10000) sont récupérés à ~2,7 % — le sens
  spécifique fuit moins.

Implication : la confidentialité à α_e=1,0 résiste sur le **contenu**
(tokens rares, porteurs du sens), mais laisse fuir la **structure
grammaticale** — cohérent avec la démonstration de la section 12 (bribes
isolées, pas de sens reconstituable).